# Train MechaDetect end to end

This notebook runs the complete one-GPU pipeline: acquire the licensed data, freeze leak-free splits, train and promote the teacher, distill **Quark** (ViT-S, 25M) and **Atom** (ViT-B, 89M), harden both with Adversarial Transformation Training (ATT), evaluate clean and transformed images, and export Float32 ONNX models.

**Requirements:** repository root as the working directory, Python 3.11, `uv`, an NVIDIA CUDA GPU, enough local storage for the image corpus and checkpoints, and `HF_TOKEN` for any gated/private Hugging Face sources. The default is the full run. Set `SMOKE_TEST = True` in the next cell to verify every training path with two updates before committing to the long run.

The organizer demonstration set (COCO val2017 + WildFake DALL-E Advanced) is never used for training.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
from pathlib import Path

ROOT = Path.cwd().resolve()
assert (ROOT / 'pyproject.toml').is_file(), 'Open this notebook from the repository root.'

DATA_ROOT = Path(os.getenv('TECHJAM_DATA_ROOT', ROOT / 'data')).resolve()
OUTPUT_ROOT = Path(os.getenv('TECHJAM_OUTPUT_ROOT', ROOT / 'outputs')).resolve()
HF_HOME = Path(os.getenv('TECHJAM_HF_HOME', ROOT / '.cache' / 'huggingface')).resolve()
DATASET_REPO = 'zye2/tj-data'
DATASET_REVISION = 'e38715a99268236b1c91ac649c38fc31a3d39867'
DATA_PACKAGE = ROOT / '.runtime' / 'tj-data'
MANIFEST_SOURCE = DATA_PACKAGE / 'production-2026-08-31' / 'manifests'
MANIFEST_DIR = ROOT / 'splits' / 'production_eligible'
NUM_WORKERS = min(8, os.cpu_count() or 4)
SMOKE_TEST = False  # True: two training updates per stage; False: complete training

for directory in (DATA_ROOT, OUTPUT_ROOT, HF_HOME, DATA_PACKAGE, MANIFEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

ENV = os.environ.copy()
ENV.update({
    'TECHJAM_DATA_ROOT': str(DATA_ROOT),
    'TECHJAM_OUTPUT_ROOT': str(OUTPUT_ROOT),
    'TECHJAM_HF_HOME': str(HF_HOME),
    'HF_HOME': str(HF_HOME),
})

def run(*parts: object) -> None:
    command = [str(part) for part in parts]
    print('\n$', subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, cwd=ROOT, env=ENV, check=True)

def latest_checkpoint(directory: Path) -> Path:
    candidates = [path for path in directory.glob('checkpoint-*.pt') if path.name != 'checkpoint-promoted.pt']
    if not candidates:
        raise FileNotFoundError(f'No checkpoint found in {directory}')
    return max(candidates, key=lambda path: path.stat().st_mtime)

print(json.dumps({
    'data_root': str(DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'hf_home': str(HF_HOME),
    'workers': NUM_WORKERS,
    'smoke_test': SMOKE_TEST,
}, indent=2))

## 1. Install and verify the runtime

`uv sync --locked` installs the exact dependency versions already recorded in `uv.lock`. The second command fails immediately if CUDA is unavailable.

In [ ]:
run('uv', 'sync', '--locked')
run('uv', 'run', 'python', '-c',
    "import torch; assert torch.cuda.is_available(), 'CUDA GPU required'; "
    "print(torch.cuda.get_device_name(0)); print('PyTorch', torch.__version__)")

## 2. Download manifests and acquire source images

The manifest package is pinned to an immutable Hugging Face revision. `acquire_all_images.py` downloads the declared public/licensed source records, resumes partial downloads, and blocks the organizer demonstration set and other forbidden cohorts.

In [ ]:
run(
    'uv', 'run', 'hf', 'download', DATASET_REPO,
    '--repo-type', 'dataset',
    '--revision', DATASET_REVISION,
    '--include', 'production-2026-08-31/manifests/*',
    '--local-dir', DATA_PACKAGE,
)
assert (MANIFEST_SOURCE / 'declared_manifest.parquet').is_file()
run(
    'uv', 'run', 'python', 'scripts/data_prep/acquire_all_images.py',
    '--data-root', DATA_ROOT,
    '--manifest-dir', MANIFEST_SOURCE,
    '--workers', NUM_WORKERS,
    '--resume',
    '--verify-bytes',
)

## 3. Freeze training, validation, test, and calibration splits

This step decodes every selected image, quarantines corrupt or conflicting rows, groups exact/perceptual duplicates, prevents group leakage, and excludes held-out generator families from training.

In [ ]:
run(
    'uv', 'run', 'python', 'scripts/data_prep/freeze_production_eligible.py',
    '--declared-dir', MANIFEST_SOURCE,
    '--output-dir', MANIFEST_DIR,
    '--data-root', DATA_ROOT,
    '--calibration-size', 4096,
    '--strict',
    '--verify-bytes',
)
TRAIN_MANIFEST = MANIFEST_DIR / 'train.parquet'
VALIDATION_MANIFEST = MANIFEST_DIR / 'validation.parquet'
TEST_MANIFEST = MANIFEST_DIR / 'test.parquet'
for path in (TRAIN_MANIFEST, VALIDATION_MANIFEST, TEST_MANIFEST):
    assert path.is_file(), path

## 4. Train the 872.6M-parameter teacher

Stage 1 trains the detector heads with a frozen DINOv3 ViT-H+/16 backbone. Stage 2 unfreezes the backbone, enables gradient checkpointing and EMA, and trains paired original/transformed examples. Both stages use BF16 and effective batch 48 on one GPU. A single RTX 4080 is supported; the full run is long.

In [ ]:
stage1_args = [
    'uv', 'run', 'python', '-m', 'aigc_detector.train',
    '--config', 'configs/teacher_dinov3_stage1_clean_frozen.yaml',
    '--world-size', 1, '--physical-batch-size', 6,
    '--gradient-accumulation', 8, '--num-workers', NUM_WORKERS,
]
if SMOKE_TEST:
    stage1_args += ['--max-steps', 2, '--stage', 'teacher_stage1_smoke']
run(*stage1_args)
STAGE1_DIR = OUTPUT_ROOT / ('teacher_stage1_smoke' if SMOKE_TEST else 'teacher_stage1_clean_frozen')
stage1_checkpoint = latest_checkpoint(STAGE1_DIR)
print('Stage 1 checkpoint:', stage1_checkpoint)

In [ ]:
stage2_args = [
    'uv', 'run', 'python', '-m', 'aigc_detector.train',
    '--config', 'configs/teacher_dinov3_stage2_paired_unfrozen.yaml',
    '--initial-checkpoint', stage1_checkpoint,
    '--world-size', 1, '--physical-batch-size', 2,
    '--gradient-accumulation', 24, '--num-workers', NUM_WORKERS,
]
if SMOKE_TEST:
    stage2_args += ['--max-steps', 2, '--stage', 'teacher_stage2_smoke']
run(*stage2_args)
STAGE2_DIR = OUTPUT_ROOT / ('teacher_stage2_smoke' if SMOKE_TEST else 'teacher_stage2_paired_unfrozen')
stage2_checkpoint = latest_checkpoint(STAGE2_DIR)
print('Stage 2 checkpoint:', stage2_checkpoint)

## 5. Evaluate and promote the teacher

A full run calibrates the operating threshold on validation data and only writes `checkpoint-promoted.pt` if the quality gate passes. Smoke mode intentionally skips this expensive gate and carries its two-step checkpoint into the remaining smoke paths.

In [ ]:
TEACHER_REPORT = STAGE2_DIR / 'promotion_report.json'
TEACHER_METADATA = STAGE2_DIR / 'metadata.json'
TEACHER_PROMOTED = STAGE2_DIR / 'checkpoint-promoted.pt'
if not SMOKE_TEST:
    run(
        'uv', 'run', 'python', 'scripts/promote_teacher.py',
        '--checkpoints', stage2_checkpoint,
        '--manifest', VALIDATION_MANIFEST,
        '--config', 'configs/teacher_dinov3_stage2_paired_unfrozen.yaml',
        '--data-root', DATA_ROOT,
        '--output-report', TEACHER_REPORT,
        '--output-metadata', TEACHER_METADATA,
        '--output-checkpoint', TEACHER_PROMOTED,
        '--batch-size', 1, '--device', 'cuda',
    )
    assert TEACHER_PROMOTED.is_file(), 'Teacher did not pass its promotion gate.'
else:
    TEACHER_PROMOTED = stage2_checkpoint
print('Teacher for distillation:', TEACHER_PROMOTED)

## 6. Distill Quark and Atom sequentially

Quark uses DINOv3 ViT-S (25.1M parameters); Atom uses DINOv3 ViT-B (89.4M). Sequential execution keeps the one-GPU memory contract simple. The frozen teacher is compiled once per run with `torch.compile(mode='reduce-overhead')`.

In [ ]:
STUDENTS = {
    'quark': {'variant': 'small', 'config': 'configs/student_dinov3_small_distill.yaml', 'batch': 12, 'accum': 4},
    'atom': {'variant': 'base', 'config': 'configs/student_dinov3_base_distill.yaml', 'batch': 3, 'accum': 16},
}
for family, spec in STUDENTS.items():
    output = OUTPUT_ROOT / f'{family}_distilled'
    args = [
        'uv', 'run', 'python', 'scripts/distill_student.py',
        '--teacher-config', 'configs/teacher_dinov3_stage2_paired_unfrozen.yaml',
        '--teacher-checkpoint', TEACHER_PROMOTED,
        '--manifest', TRAIN_MANIFEST, '--val-manifest', VALIDATION_MANIFEST,
        '--output-dir', output, '--student', spec['variant'],
        '--student-config', spec['config'], '--world-size', 1,
        '--physical-batch-size', spec['batch'],
        '--gradient-accumulation', spec['accum'], '--num-workers', NUM_WORKERS,
    ]
    if SMOKE_TEST:
        args += ['--skip-teacher-gate', '--dry-run']
    else:
        args += ['--teacher-promotion-report', TEACHER_REPORT]
    run(*args)
    spec['distilled_dir'] = output
    spec['distilled_checkpoint'] = output / ('checkpoint-final.pt' if SMOKE_TEST else 'checkpoint-promoted.pt')
    assert spec['distilled_checkpoint'].is_file(), spec['distilled_checkpoint']

## 7. Adversarial Transformation Training

ATT samples three candidate transformations per row, scores them without gradient storage, and backpropagates through the hardest transformed view plus the original.

In [ ]:
for family, spec in STUDENTS.items():
    output = OUTPUT_ROOT / f'{family}_att'
    args = [
        'uv', 'run', 'python', 'scripts/train_att.py',
        '--variant', spec['variant'], '--student-checkpoint', spec['distilled_checkpoint'],
        '--manifest', TRAIN_MANIFEST,
        '--config', f"configs/att_student_{spec['variant']}.yaml",
        '--output-dir', output, '--num-candidates', 3, '--epochs', 1,
        '--world-size', 1,
    ]
    if SMOKE_TEST:
        args.append('--dry-run')
    run(*args)
    spec['att_dir'] = output
    spec['att_checkpoint'] = output / 'checkpoint-final.pt'
    assert spec['att_checkpoint'].is_file(), spec['att_checkpoint']

## 8. Clean and transformed evaluation

The complete run evaluates each distilled and ATT model on the frozen validation split. `--robustness` covers the organizer-aligned JPEG, blur, resize, noise, color, and crop grid. The ATT gate compares the before/after reports and promotes both hardened checkpoints only if they preserve clean quality and improve robustness.

In [ ]:
if not SMOKE_TEST:
    for family, spec in STUDENTS.items():
        spec['float_eval'] = OUTPUT_ROOT / f'{family}_distilled_eval.json'
        spec['att_eval'] = OUTPUT_ROOT / f'{family}_att_eval.json'
        for checkpoint, config, output in (
            (spec['distilled_checkpoint'], spec['config'], spec['float_eval']),
            (spec['att_checkpoint'], f"configs/att_student_{spec['variant']}.yaml", spec['att_eval']),
        ):
            run(
                'uv', 'run', 'python', 'scripts/evaluate_performance.py',
                '--manifest', VALIDATION_MANIFEST, '--checkpoint', checkpoint,
                '--config', config, '--output', output,
                '--robustness', '--batch-size', 8,
            )
    run(
        'uv', 'run', 'python', 'scripts/check_att_gate.py',
        '--small-float-eval', STUDENTS['quark']['float_eval'],
        '--small-att-eval', STUDENTS['quark']['att_eval'],
        '--base-float-eval', STUDENTS['atom']['float_eval'],
        '--base-att-eval', STUDENTS['atom']['att_eval'],
        '--small-checkpoint', STUDENTS['quark']['att_checkpoint'],
        '--base-checkpoint', STUDENTS['atom']['att_checkpoint'],
        '--small-output-dir', STUDENTS['quark']['att_dir'],
        '--base-output-dir', STUDENTS['atom']['att_dir'],
        '--shared-report', OUTPUT_ROOT / 'att_promotion_report.json',
        '--promote',
    )
    for spec in STUDENTS.values():
        spec['promoted_checkpoint'] = spec['att_dir'] / 'checkpoint-promoted.pt'
else:
    for spec in STUDENTS.values():
        spec['promoted_checkpoint'] = spec['att_checkpoint']

## 9. Export Float32 ONNX models

Export runs PyTorch-versus-ONNX numerical parity before writing each artifact. The generated files remain under `outputs/models/`; the repository intentionally does not track model weights.

In [ ]:
MODEL_DIR = OUTPUT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
manifest_digest = hashlib.sha256(TRAIN_MANIFEST.read_bytes()).hexdigest()
for family, spec in STUDENTS.items():
    output = MODEL_DIR / f'mechadetect-{family}-normal-post-att-float32.onnx'
    args = [
        'uv', 'run', 'python', 'scripts/export_onnx_webgpu.py',
        '--checkpoint', spec['promoted_checkpoint'], '--variant', spec['variant'],
        '--stage', 'normal_post_att', '--config', f"configs/att_student_{spec['variant']}.yaml",
        '--output', output,
    ]
    report = spec['att_dir'] / 'promotion_report.json'
    if report.is_file():
        args += ['--promotion-report', report]
    else:
        args += [
            '--calibrated-threshold', 0.5, '--manifest-digest', manifest_digest,
            '--evaluation-status', 'experimental',
        ]
    run(*args)
    spec['onnx'] = output
print('Exports:')
for family, spec in STUDENTS.items():
    print(f"  {family}: {spec['onnx']} ({spec['onnx'].stat().st_size / 1024**2:.1f} MiB)")

## 10. Inspect the outputs

A complete run should end with two promoted ATT checkpoints, two promotion reports, evaluation JSON, and two parity-verified ONNX models. For submission-format inference, run `uv run python predict.py --input <image-directory> --output predictions.json`.

In [ ]:
summary = {
    family: {
        'checkpoint': str(spec['promoted_checkpoint']),
        'onnx': str(spec['onnx']),
    }
    for family, spec in STUDENTS.items()
}
print(json.dumps(summary, indent=2))